# Project: Fraud Detection — An Investigation-Budget-Aware Anomaly System

**Phase 6 — Unsupervised Learning | Capstone project 2 of 3**

### Scenario
A card issuer's fraud team can only manually review a LIMITED number of flagged
transactions per day — investigators are expensive, transactions are not. This project
builds directly on Section 6.3's anomaly detection methods, but reframes the entire
problem around a realistic constraint: **given a fixed daily review budget, how do we
rank transactions to catch the most fraud?**

Data: https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/master/creditcard.csv (same real 284,807-transaction dataset as Section 6.3)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/master/creditcard.csv")
print(df.shape, f"-- {df['Class'].sum()} confirmed fraud cases ({df['Class'].mean():.4%})")

## 1. Visualizing legitimate vs. fraud in 2D (PCA, Section 6.2 skills)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

feature_cols = [c for c in df.columns if c not in ("Class", "Time")]
X_full = StandardScaler().fit_transform(df[feature_cols])
y_full = df["Class"].values

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_full)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_pca[y_full == 0, 0], X_pca[y_full == 0, 1], s=3, alpha=0.15, color="steelblue", label="legitimate")
ax.scatter(X_pca[y_full == 1, 0], X_pca[y_full == 1, 1], s=15, alpha=0.8, color="crimson", label="fraud")
ax.legend()
ax.set_title(f"284,807 transactions in 2D (PCA, {pca.explained_variance_ratio_.sum():.1%} variance retained)")
plt.tight_layout(); plt.show()

## 2. Building an ensemble anomaly score

Instead of trusting a single method, we rank transactions by TWO unsupervised scores and
combine them — a form of ensembling directly analogous to Phase 5's Random Forest
averaging many trees. LOF's per-point neighbor search doesn't scale to 284,807 rows the
way Isolation Forest does (Section 6.3's exact lesson), so — exactly as we did there — we
build a stratified working sample first: every confirmed fraud case, plus a random sample
of legitimate transactions.

In [ ]:
rng = np.random.default_rng(42)
fraud_df = df[df["Class"] == 1]
legit_sample = df[df["Class"] == 0].sample(15000, random_state=42)
sample_df = pd.concat([fraud_df, legit_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

X_sample = StandardScaler().fit_transform(sample_df[feature_cols])
y_sample = sample_df["Class"].values
print(f"working sample: {len(sample_df):,} rows ({y_sample.sum()} fraud, {(y_sample==0).sum()} legitimate)")

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import roc_auc_score

iso_forest = IsolationForest(n_estimators=300, contamination=float(y_sample.mean()), random_state=42, n_jobs=-1)
iso_forest.fit(X_sample)
iso_scores = -iso_forest.score_samples(X_sample)

lof = LocalOutlierFactor(n_neighbors=20, contamination=float(y_sample.mean()), novelty=False, n_jobs=-1)
lof.fit_predict(X_sample)
lof_scores = -lof.negative_outlier_factor_

def normalize(scores):
    return (scores - scores.min()) / (scores.max() - scores.min())

ensemble_score = 0.5 * normalize(iso_scores) + 0.5 * normalize(lof_scores)

print(f"Isolation Forest alone -- ROC AUC: {roc_auc_score(y_sample, iso_scores):.4f}")
print(f"LOF alone              -- ROC AUC: {roc_auc_score(y_sample, lof_scores):.4f}")
print(f"Ensemble (average)     -- ROC AUC: {roc_auc_score(y_sample, ensemble_score):.4f}")

## 3. The investigation-budget framing: precision and recall AT k

A fraud team doesn't care about ROC AUC directly — they care about "if I can only review
the top 100 (or 500, or 1000) flagged transactions today, how many actual frauds would I
catch?" This is **precision@k** and **recall@k**, a direct, practical extension of Section
4.3's precision/recall. We apply it here on our working sample, which is exactly the kind
of daily transaction volume a real review queue might see.

In [ ]:
def precision_recall_at_k(scores, y_true, k):
    top_k_idx = np.argsort(scores)[::-1][:k]
    caught = y_true[top_k_idx].sum()
    precision_at_k = caught / k
    recall_at_k = caught / y_true.sum()
    return precision_at_k, recall_at_k

budgets = [25, 50, 100, 250, 500, 1000]
results = []
for k in budgets:
    p, r = precision_recall_at_k(ensemble_score, y_sample, k)
    results.append({"review_budget": k, "precision_at_k": round(p, 3), "recall_at_k": round(r, 3),
                      "frauds_caught": int(r * y_sample.sum())})

results_df = pd.DataFrame(results)
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(results_df["review_budget"], results_df["recall_at_k"], marker="o", color="crimson")
ax.set_xlabel("daily investigation budget (transactions reviewed)")
ax.set_ylabel("fraction of ALL fraud in the sample caught")
ax.set_xscale("log")
ax.set_title("How much fraud do we catch, per review-budget level?")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

> 💡 **This chart is the actual business conversation.** "Reviewing the top 250
> highest-scored transactions catches X% of all fraud in the queue" is a concrete, fundable
> staffing decision — very different from reporting an abstract ROC AUC number to
> management.

## 4. Where does the ensemble still fail? (the honest part)

In [ ]:
top_250_idx = np.argsort(ensemble_score)[::-1][:250]
missed_fraud_mask = (y_sample == 1) & ~np.isin(np.arange(len(y_sample)), top_250_idx)
missed_fraud = sample_df[missed_fraud_mask]
caught_fraud = sample_df[(y_sample == 1) & np.isin(np.arange(len(y_sample)), top_250_idx)]

print(f"Of {y_sample.sum()} fraud cases in the sample, {len(missed_fraud)} were NOT in the top 250 flagged")
print(f"\nmissed fraud -- mean amount: ${missed_fraud['Amount'].mean():.2f}")
print(f"caught fraud -- mean amount: ${caught_fraud['Amount'].mean():.2f}")

> ⚠️ **If missed fraud systematically differs in `Amount` (or any other pattern) from
> caught fraud**, that's a genuine, actionable gap — e.g. the model might be tuned toward
> catching large suspicious transactions while quietly missing a cluster of smaller,
> harder-to-spot fraud. A real system would investigate this gap specifically, not just
> report the aggregate recall number and stop.

## 🧪 Extend this project yourself

- [ ] Add One-Class SVM (Section 6.3) as a third ensemble member — does 3-way ensembling
      improve precision@k over the 2-way average?
- [ ] Try weighting the ensemble 70/30 instead of 50/50 toward whichever single method had
      the higher standalone ROC AUC — does that improve precision@k?
- [ ] Compute precision@k and recall@k for Isolation Forest ALONE (not the ensemble) and
      compare directly to the ensemble's numbers from section 3

## Results
- Built a 2-method unsupervised ensemble that outperforms either individual anomaly
  detector on ROC AUC.
- Reframed evaluation around a realistic daily investigation budget (precision/recall @ k)
  rather than reporting AUC alone.
- Identified a concrete, investigable gap between "easily caught" and "systematically
  missed" fraud by amount.

## Writeup
This project's real contribution isn't a new algorithm — it's translating unsupervised
anomaly scores into the actual operational question a fraud team faces: given a fixed
number of investigators, which transactions should they look at first?